<!-- NB_DOC_INTRO_V1 -->
# NLP — LangChain et RAG end-to-end

> 📚 **Doc thematique** : [docs/05_NLP.md](docs/05_NLP.md) (NLP)
> 📖 **Inventaire** : [docs/INVENTAIRE.md](docs/INVENTAIRE.md) | 🗂️ **README** : [README.md](README.md)

---

## Presentation

**LangChain** orchestre des **LLMs** avec d'autres outils (vector stores, APIs, BDD, tools). Le pattern phare : **RAG** (Retrieval-Augmented Generation) — recuperer du contexte pertinent + le passer au LLM pour ancrer ses reponses dans des sources, **anti-hallucination**.

Ce notebook construit un **RAG complet from-scratch** avec sentence-transformers (embeddings locaux) + sklearn cosine similarity (vector store minimal). **Code execute** sans dependance lourde (langchain optionnel). Pour la version langchain : code commente pret a executer.

Pourquoi LangChain ? Quand le RAG simple ne suffit plus : memoire conversationnelle, agents avec tools, evaluation systematique, observability. Mais **critique honnete** : pour un POC, ~ 100 lignes Python suffisent (cf. cellules executees).

Versions : `sentence-transformers >= 2.5`, `langchain >= 0.1` (optionnel).

## Plan

1. Architecture RAG (loaders → splitters → embeddings → vector store → retriever → LLM)
2. **Demo from-scratch executable** : mini-RAG avec sentence-transformers + cosine
3. Chunking strategies (fixed-size, semantic, hierarchical)
4. Embeddings : choix de modele 2024-2025 (MTEB leaderboard)
5. Vector stores : in-memory, FAISS, Chroma, Qdrant (comparatif)
6. Re-ranking avec cross-encoder
7. Hybrid search (BM25 + dense + RRF)
8. **LangChain** : composants + LCEL (Expression Language)
9. Memory pour chat (conversationnel)
10. Agents et Tools
11. Evaluation RAG (ragas, LLM-as-judge)
12. Production (streaming, callbacks, caching)
13. Pieges et anti-patterns
14. References


## 1. Architecture RAG

```
Documents
   │
   ▼
[Loaders]      ← PDF, HTML, Notion, Slack, ...
   │
   ▼
[Splitters]    ← chunks (taille + overlap)
   │
   ▼
[Embedder]     ← sentence-transformers, OpenAI ada, BGE, E5
   │
   ▼
Vecteurs (N × d)
   │
   ▼
[Vector store] ← FAISS (in-memory), Chroma (local), Qdrant (server)
   │
   ▼
Query → embed(query) → top-k retrieval → [docs candidats]
                                            │
                                            ▼
                                  [Re-ranker (cross-encoder)] ← optionnel mais boost +10pts
                                            │
                                            ▼
                                  Top-N final → LLM (avec context = top-N)
```


## 2. Mini-RAG executable from-scratch


In [ ]:
# pip install -q sentence-transformers scikit-learn numpy
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import warnings
warnings.filterwarnings("ignore")

# === 1. Corpus jouet ===
documents = [
    "Le RAG combine un retriever (recherche semantique) et un generateur (LLM).",
    "Les embeddings transformer convertissent du texte en vecteurs denses.",
    "FAISS (Meta) est une bibliotheque pour la recherche de plus proches voisins.",
    "Sentence-Transformers fournit des modeles pretrained pour embeddings de phrases.",
    "Le chunking (decoupage) est crucial : trop petit = perte de contexte, trop grand = retrieval imprecis.",
    "BM25 est un algorithme de recherche lexicale (TF-IDF ameliore), souvent baseline robuste.",
    "Le re-ranking avec cross-encoder ameliore generalement le top-k de 10-20% NDCG.",
    "Python est un langage de programmation de haut niveau.",
    "Les LLMs hallucinent quand ils n'ont pas l'information : RAG attenue ce probleme.",
    "ChromaDB et Qdrant sont des bases vectorielles populaires en 2024.",
]
print(f"Corpus : {len(documents)} documents")

In [ ]:
# === 2. Embedder (modele leger 80 MB, EN) ===
print("Loading embeddings model (peut prendre 30s premiere fois)...")
embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
print(f"Modele : {embedder._first_module().auto_model.config.architectures}")
print(f"Dim embeddings : {embedder.get_sentence_embedding_dimension()}")

In [ ]:
# === 3. Encoder le corpus ===
doc_embeddings = embedder.encode(documents, show_progress_bar=False)
print(f"Shape : {doc_embeddings.shape}")  # (N_docs, 384)

In [ ]:
# === 4. Retrieval = cosine similarity entre query et corpus ===
def retrieve(query: str, k: int = 3) -> list[tuple[str, float]]:
    q_emb = embedder.encode([query], show_progress_bar=False)
    sims = cosine_similarity(q_emb, doc_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:k]
    return [(documents[i], float(sims[i])) for i in top_idx]

# === 5. Test ===
queries = [
    "Qu'est-ce que le RAG ?",
    "Comment fonctionne la recherche de similarite ?",
    "Quelle base vectorielle utiliser ?",
]
for q in queries:
    print(f"\nQuery : {q}")
    for doc, sim in retrieve(q, k=2):
        print(f"  [{sim:.3f}] {doc}")

**Lecture** : on a un mini-RAG en ~ 30 lignes. Etape suivante : ajouter un LLM qui prend les top-k comme contexte et synthese la reponse.

### 2.1 Avec LLM (mock + pattern complet)


In [ ]:
def build_rag_prompt(question: str, retrieved_docs: list[str]) -> str:
    context = "\n\n".join(f"[doc {i+1}] {d}" for i, d in enumerate(retrieved_docs))
    return f'''Tu es un assistant qui repond UNIQUEMENT en t'appuyant sur le contexte fourni.
Si la reponse n'est pas dans le contexte, dis : "Je ne sais pas".

Contexte :
{context}

Question : {question}

Reponse (cite les doc entre crochets) :'''

# Demo avec mock LLM (pour exec sans API)
def mock_llm(prompt: str) -> str:
    return "[Reponse simulee — un vrai LLM utiliserait le contexte pour synthese]"

q = "Qu'est-ce que le RAG et pourquoi l'utiliser ?"
top_docs = [d for d, _ in retrieve(q, k=3)]
prompt = build_rag_prompt(q, top_docs)
answer = mock_llm(prompt)
print(f"Q : {q}")
print(f"\nPrompt construit (~ {len(prompt)} chars) :")
print(prompt[:400], "...")
print(f"\nA : {answer}")
print(f"\n→ Pour appel reel : utiliser Ollama (local) ou OpenAI/Anthropic (API)")

## 3. Chunking strategies

Le chunking est **critique** : trop petit = perte de contexte (un chunk = 1 phrase sans le paragraphe d'avant), trop grand = retrieval imprecis (un chunk de 5000 chars qui contient 1 info utile + 4990 chars de bruit).

| Strategie | Description | Quand |
|---|---|---|
| **Fixed-size** | Decouper a N caracteres avec overlap | Baseline simple |
| **Sentence-based** | Couper aux fins de phrases | Preserve la structure |
| **Recursive** (LangChain default) | Essaie `\n\n`, puis `\n`, puis `. `, etc. | Adapte au texte |
| **Semantic** | Embeddings + detection de rupture de sens | Experimental, lent |
| **Hierarchical** | Section → Paragraph → Sentence, retrieve parent ou enfant | Pour structure document forte (livres) |
| **MarkdownHeader-based** | Decoupe aux `#`, `##`, etc. | Documents markdown/notion |

### Implementation simple : recursive


In [ ]:
def recursive_chunk(text: str, chunk_size: int = 200, overlap: int = 30) -> list[str]:
    '''Decoupage recursif avec essai de separateurs naturels.'''
    if len(text) <= chunk_size:
        return [text]

    separators = ["\n\n", "\n", ". ", " ", ""]
    chunks = []
    for sep in separators:
        if sep in text:
            parts = text.split(sep)
            current = ""
            for part in parts:
                if len(current) + len(part) + len(sep) <= chunk_size:
                    current = current + sep + part if current else part
                else:
                    if current:
                        chunks.append(current.strip())
                    # Avec overlap : recommencer en gardant la fin
                    if overlap > 0 and chunks:
                        current = chunks[-1][-overlap:] + sep + part
                    else:
                        current = part
            if current:
                chunks.append(current.strip())
            return [c for c in chunks if c]
    return [text]   # fallback

long_text = " ".join(documents) * 3
chunks = recursive_chunk(long_text, chunk_size=200, overlap=30)
print(f"Texte original : {len(long_text)} chars")
print(f"Nb chunks      : {len(chunks)}")
print(f"Premier chunk  : {chunks[0][:100]}...")

## 4. Embeddings — choix de modele 2024-2025

Standard : **MTEB** leaderboard (https://huggingface.co/spaces/mteb/leaderboard).

| Modele | Dims | Langue | Taille | MTEB score | Quand |
|---|---:|---|---:|---:|---|
| `sentence-transformers/all-MiniLM-L6-v2` | 384 | EN | 80 MB | 56.3 | Baseline rapide, dev |
| `BAAI/bge-large-en-v1.5` | 1024 | EN | 1.3 GB | 64.2 | Top EN, prod |
| `intfloat/e5-large-v2` | 1024 | EN | 1.3 GB | 62.3 | Top EN, alternative |
| `intfloat/multilingual-e5-large` | 1024 | Multi (FR ok) | 2.2 GB | 64.4 | Multi-langues |
| `BAAI/bge-m3` | 1024 | Multi | 2.3 GB | excellent | Multi + flexible (dense/sparse/multi-vector) |
| `jinaai/jina-embeddings-v3` | 1024 | Multi | 700 MB | 65.5 | Top 2024, long context (8k) |
| `openai/text-embedding-3-large` | 3072 | Multi | API | 64.6 | SaaS, qualite top |
| `openai/text-embedding-3-small` | 1536 | Multi | API | 62.3 | SaaS, prix bas |


## 5. Vector stores

| Store | Type | Forces | Limites |
|---|---|---|---|
| **In-memory numpy** | DIY | Trivial, transparent | Pas scale > 10k docs |
| **FAISS** | In-process | Tres rapide, ANN multiple algos | Pas de filtres metadata natifs |
| **Chroma** | In-process / Server | Pythonic, langchain integration | Plus jeune, scale moyen |
| **Qdrant** | Serveur Rust | Filtres puissants, payload riche, REST+gRPC | Serveur a deployer |
| **Pinecone** | SaaS | Zero-ops, scale | Vendor lock-in, cout |
| **Milvus** | Serveur distribue | Scale billions | Complexe a operer |
| **Weaviate** | Serveur | Schema + hybrid search natif | Plus lourd |
| **pgvector** | Postgres extension | Si deja Postgres en prod | Perf moindre que dedies |


## 6. Re-ranking avec cross-encoder

**Bi-encoder** (sentence-transformer classique) : encode docs et query SEPAREMENT → rapide, indexable. Mais qualite limitee.
**Cross-encoder** : encode `(query, doc)` ENSEMBLE → tres precis mais O(N) au runtime.

**Pattern combine** : bi-encoder pour top-100, puis cross-encoder pour top-3 final. Boost typique : +5-15 pts NDCG.

```python
# pip install -q sentence-transformers
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

# Apres avoir recupere top-100 via bi-encoder :
pairs = [(query, doc) for doc in top_100_docs]
scores = reranker.predict(pairs)
top_10 = sorted(zip(top_100_docs, scores), key=lambda x: -x[1])[:10]
```


## 7. Hybrid search (BM25 + dense + RRF)

**BM25** (lexical) detecte les correspondances **exactes de mots-cles**. **Dense embeddings** detectent la **semantique**. Combiner les deux via **Reciprocal Rank Fusion (RRF)** donne souvent le meilleur recall.

```python
# pip install -q rank-bm25
from rank_bm25 import BM25Okapi

# BM25
tokenized_corpus = [doc.lower().split() for doc in documents]
bm25 = BM25Okapi(tokenized_corpus)
bm25_scores = bm25.get_scores(query.lower().split())
bm25_ranks = sorted(range(len(documents)), key=lambda i: -bm25_scores[i])

# Dense (deja fait plus haut)
dense_ranks = sorted(range(len(documents)),
                     key=lambda i: -cosine_similarity([q_emb], [doc_embeddings[i]])[0][0])

# RRF
def rrf(rankings, k=60):
    scores = {}
    for ranking in rankings:
        for rank, doc_id in enumerate(ranking, 1):
            scores[doc_id] = scores.get(doc_id, 0) + 1 / (k + rank)
    return sorted(scores.items(), key=lambda x: -x[1])

fused = rrf([bm25_ranks, dense_ranks])
top_5_fused = [documents[i] for i, _ in fused[:5]]
```


## 8. LangChain — composants + LCEL

LangChain abstrait : `Loader` → `Splitter` → `Embedder` → `VectorStore` → `Retriever` → `LLM` → `OutputParser`. Tout est composable via **LCEL** (LangChain Expression Language, syntaxe `|` style Unix pipe).

```python
# pip install -q langchain langchain-community langchain-openai langchain-huggingface langchain-chroma chromadb

from langchain_community.document_loaders import PyPDFLoader, TextLoader, WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

# === 1. Load ===
loader = PyPDFLoader("article.pdf")
docs = loader.load()

# === 2. Split ===
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""],
)
chunks = splitter.split_documents(docs)

# === 3. Embed + index ===
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vectorstore = Chroma.from_documents(chunks, embeddings, persist_directory="./chroma_db")

# === 4. Retriever ===
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})

# === 5. LLM + Prompt + Chain (LCEL) ===
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

prompt = ChatPromptTemplate.from_template('''
Tu es un assistant qui repond UNIQUEMENT en t'appuyant sur le contexte.
Si tu ne sais pas, dis "Je ne sais pas".

Contexte : {context}
Question : {question}
Reponse :
''')

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

answer = rag_chain.invoke("Quelle est la conclusion principale du document ?")
print(answer)
```


## 9. Memory pour chat (conversationnel)

Sans memory, chaque appel est independant. Avec, le LLM "se souvient" des echanges precedents.

```python
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

histories = {}
def get_history(session_id):
    if session_id not in histories:
        histories[session_id] = ChatMessageHistory()
    return histories[session_id]

rag_with_history = RunnableWithMessageHistory(
    rag_chain, get_history,
    input_messages_key="question",
    history_messages_key="history",
)

config = {"configurable": {"session_id": "user_42"}}
rag_with_history.invoke({"question": "Bonjour"}, config)
rag_with_history.invoke({"question": "Et puis ?"}, config)  # se souvient
```


## 10. Agents et Tools

Un **agent** = un LLM qui peut **decider** d'appeler des outils (search, calculator, code, etc.).

```python
from langchain.agents import AgentExecutor, create_react_agent
from langchain.tools import tool
from langchain_experimental.tools.python.tool import PythonREPLTool
from langchain import hub

@tool
def get_weather(city: str) -> str:
    '''Renvoie la meteo pour une ville donnee.'''
    return f"Sunny, 22°C in {city}"

tools = [get_weather, PythonREPLTool()]

# ReAct = Reasoning + Acting (Yao et al. 2022)
prompt = hub.pull("hwchase17/react")
agent = create_react_agent(llm, tools, prompt)
executor = AgentExecutor(agent=agent, tools=tools, verbose=True)
executor.invoke({"input": "Quel temps a Paris, et calcule 2 + 2"})
```

> ⚠️ Les agents sont **chers** (multi-appels LLM) et **fragiles** (boucles infinies). Utiliser avec parcimonie + max_iterations.


## 11. Evaluation RAG

[ragas](https://docs.ragas.io/) — metriques cles :
- **Faithfulness** : la reponse est-elle ancree dans le contexte ? (anti-hallucination)
- **Answer relevancy** : la reponse repond-elle a la question ?
- **Context precision** : les chunks recuperes sont-ils tous pertinents ?
- **Context recall** : tous les chunks pertinents ont-ils ete recuperes ?

```python
# pip install -q ragas datasets
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

data = {
    "question":     ["Quelle est la capitale de la France ?"],
    "answer":       ["La capitale de la France est Paris."],
    "contexts":     [["Paris est la capitale et plus grande ville de la France."]],
    "ground_truth": ["Paris"],
}
ds = Dataset.from_dict(data)
result = evaluate(ds, metrics=[faithfulness, answer_relevancy, context_precision, context_recall])
print(result)
# {'faithfulness': 0.95, 'answer_relevancy': 0.92, ...}
```


## 12. Production : streaming, callbacks, observability

```python
# Streaming (afficher token par token)
async for chunk in rag_chain.astream("Ma question ?"):
    print(chunk, end="", flush=True)

# Callbacks (logger chaque etape)
from langchain.callbacks import StdOutCallbackHandler
rag_chain.invoke("...", config={"callbacks": [StdOutCallbackHandler()]})

# Caching (reduire les couts LLM)
from langchain.globals import set_llm_cache
from langchain.cache import SQLiteCache
set_llm_cache(SQLiteCache(database_path=".langchain.db"))
# Desormais les appels LLM identiques sont mis en cache
```

**Observability** :
- **LangSmith** (LangChain) — traces, replays, eval, dashboards
- **Helicone**, **Langfuse** — alternatives open-source
- **OpenTelemetry** : `langchain-otel` pour Datadog/Honeycomb


## 13. Pieges et anti-patterns

| ❌ Anti-pattern | ✅ Correction |
|---|---|
| Chunks trop petits (200 chars) | Cible 500-1000 chars, overlap 10-20% |
| Pas de overlap entre chunks | Au moins 10% (gere les debordements) |
| Embedder sans tuning sur le domaine | Modele generaliste OK pour 80% des cas |
| Top-k = 50 → context blow-up | Re-ranking, garde top-3 final |
| Pas de "say I don't know" dans le prompt | Hallucinations garanties |
| Temperature 0.7 sur question factuelle | 0 ou 0.1 |
| Pas de citation sources dans output | Anti-hallucination UX = mettre [doc 1] etc. |
| Pas d'evaluation systematique | ragas + dataset de QA gold |
| LangChain pour POC simple | ~ 100 lignes Python suffisent, ne pas surengineering |
| Persister Chroma sans `persist_directory` | Perdu au restart |
| Mauvais chat template (instruct model) | Verifier le template natif du modele |
| Inclure le prompt complet a chaque appel sans cache | Cout × 10 evitable |
| Embeddings recalculees a chaque ingestion | Persister + incremental updates |

## 14. References

- **LangChain** : https://python.langchain.com/docs/
- **LlamaIndex** : https://docs.llamaindex.ai/  (alternative orientee documents)
- **Haystack** (deepset) : https://docs.haystack.deepset.ai/
- **DSPy** : https://dspy-docs.vercel.app/  (declaratif)
- **ragas** : https://docs.ragas.io/
- **MTEB leaderboard** : https://huggingface.co/spaces/mteb/leaderboard
- **Papers** : Lewis et al. 2020 (RAG original), Karpukhin et al. 2020 (DPR), Khattab & Zaharia 2020 (ColBERT)

Voir aussi :
- [retrieval_BDD_Vectorielle.ipynb](./retrieval_BDD_Vectorielle.ipynb) — infra vector store
- [NLP_Recherche_d_informations.ipynb](./NLP_Recherche_d_informations.ipynb) — embeddings
- [BDD_Vectorielles.ipynb](./BDD_Vectorielles.ipynb) — FAISS / Weaviate
- [AI_Local_LLMs.ipynb](./AI_Local_LLMs.ipynb) — LLM local pour RAG sans API


<!-- ENRICH_2025_V1 -->

## 🔥 Alternatives a LangChain en 2024-2025

LangChain est puissant mais critiquee (verbeux, breaking changes frequents). Alternatives serieuses :

| Framework | Forces | Quand l'utiliser |
|---|---|---|
| **LlamaIndex** | Focus retrieval/RAG, abstractions plus simples | Si dominant cas = RAG sur documents |
| **Haystack 2.0** (deepset) | Production-ready, pipelines explicites | Equipes ML/IR experimentees |
| **DSPy** (Stanford) | Programmation **declarative** (signatures + optimiseurs) | Quand on cherche a optimiser systematiquement les prompts |
| **CrewAI** | Multi-agent, orchestration roles | Workflows complexes avec agents specialises |
| **AutoGen** (Microsoft) | Multi-agent conversation | Recherche / R&D agents |
| **LangGraph** (LangChain) | Graphes stateful, controls flow | Quand LangChain est utilise mais besoin orchestration fine |
| **Custom Python** (200 lignes) | Zero framework, control total | POC ou production simple |

### Exemple **DSPy** — optimisation automatique de prompt
```python
# pip install dspy-ai
import dspy

class GenerateAnswer(dspy.Signature):
    """Repond a la question en s'appuyant sur le contexte."""
    context = dspy.InputField(desc="documents pertinents")
    question = dspy.InputField()
    answer = dspy.OutputField(desc="reponse 1-3 phrases")

# Pipeline
class RAG(dspy.Module):
    def __init__(self):
        super().__init__()
        self.retrieve = dspy.Retrieve(k=5)
        self.generate = dspy.ChainOfThought(GenerateAnswer)
    def forward(self, question):
        context = self.retrieve(question).passages
        return self.generate(context=context, question=question)

# Optimisation : DSPy va auto-tune les exemples few-shot
optimizer = dspy.BootstrapFewShot(metric=my_metric, max_bootstrapped_demos=4)
optimized_rag = optimizer.compile(RAG(), trainset=train_examples)
```

DSPy **change la facon de penser** : on declare ce qu'on veut, l'optimiseur trouve les meilleurs prompts/exemples.

## 🔥 Patterns RAG avances 2025

### 1. **Hybrid Search systematique** (BM25 + dense + RRF)
Gain typique : **+10-15% NDCG** vs dense seul.

### 2. **Re-ranking cross-encoder**
Bi-encoder top-100 → cross-encoder top-3. Gain : **+5-15% NDCG**.

### 3. **HyDE** (Hypothetical Document Embeddings)
1. LLM genere une **reponse hypothetique** a la query
2. On embed cette reponse (pas la query)
3. Retrieval sur cet embedding

Souvent meilleur que retrieval sur query brute.

### 4. **Self-RAG / Corrective RAG**
Le LLM evalue si les docs recuperes sont suffisants → si non, regenere une nouvelle query.

### 5. **GraphRAG** (Microsoft, 2024)
Construire un **knowledge graph** depuis le corpus + retrieval sur graph + LLM. Tres bon pour questions multi-hop.

### 6. **Long-context RAG vs Chunks**
Avec context windows >= 128k (Claude 3.5, GPT-4 Turbo, Gemini Pro), parfois **mettre tout le doc dans le prompt** > RAG. Trade-off : cout vs precision.

## 🔥 Eval RAG en 2025 — `ragas` v0.2+

Nouvelles metriques :
- **Answer Correctness** (vs ground truth)
- **Context Entity Recall**
- **Topic Adherence**

```python
from ragas.metrics import (
    faithfulness, answer_relevancy, context_precision, context_recall,
    answer_correctness, context_entity_recall,
)
```

## 💡 Recap quand quoi

| Cas | Stack |
|---|---|
| POC rapide (1 PDF, 1 user) | sentence-transformers + FAISS + LLM (sans framework) |
| App equipe (10-100 docs) | Chroma + LangChain ou LlamaIndex |
| Production (1000+ users) | Qdrant + LangChain LCEL + LangSmith |
| Optimisation systematique | DSPy |
| Multi-agent / workflows complexes | LangGraph / CrewAI |
| Cas regulatoire / explication | Custom (control total) |
